In [1]:
!python3 -m venv venv
!source venv/bin/activate
%pwd
384

from platform import python_version

print(python_version())

3.11.2


In [3]:
!pip3 -q install db-dtypes
!pip3 -q install "google-cloud-bigquery>=3.17"
!pip3 -q install "google-cloud-aiplatform>=1.38"
!pip3 -q install "pandas>=2.2.0"

In [4]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=6vO411KbrKC9UZCjd5cpm1b0MME0Dc&access_type=offline&code_challenge=PQdONi1O0r0VMTNSkAVFg6fehUBxH4zO3kNc7AaXlzA&code_challenge_method=S256


Credentials saved to file: [/home/brendanhills/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "uk-bh-experiments-argolis" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [5]:

PROJECT_ID = "uk-bh-experiments-argolis"  # @param {type:"string"}
REGION = "US"  # @param {type: "string"}
DATASET_ID = "schema_mapping"  # @param {type:"string"}

In [6]:
import pandas as pd
from google.cloud import bigquery
from vertexai.language_models import TextEmbeddingModel


2024-02-22 17:13:11.851877: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-02-22 17:13:11.900803: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-22 17:13:11.900857: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-02-22 17:13:11.902480: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-02-22 17:13:11.910430: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-02-22 17:13:11.911304: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [7]:

REGION = "US"  # @param {type: "string"}

In [8]:
DATASET_ID = "schema_mapping"  # @param {type:"string"}

In [9]:
def get_table_sample(table_name):
  client = bigquery.Client()
  table_query = f"""
  SELECT * FROM {PROJECT_ID}.{DATASET_ID}.{table_name} TABLESAMPLE SYSTEM (10 PERCENT)
  """
  table_sample = client.query(table_query)
  table_sample_df = table_sample.to_dataframe()
  return table_sample_df




In [10]:
TABLENAME1="insurance"

table1_df = get_table_sample(TABLENAME1)

table1_df.head()



,age,sex,bmi,children,smoker,region,charges
0,18,female,26.315,0,False,northeast,2198.18985
1,18,female,38.665,2,False,northeast,3393.35635
2,18,female,35.625,0,False,northeast,2211.13075
3,18,female,30.115,0,False,northeast,21344.84670
4,18,male,23.750,0,False,northeast,1705.62450


In [11]:
TABLENAME2="df1_loan"

table2_df = get_table_sample(TABLENAME2)

table2_df.head()


,int64_field_0,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Total_Income
0,63,LP001213,Male,True,1,Graduate,False,4945,0.0,NaN,360.0,0.0,Rural,False,4945.0
1,127,LP001449,Male,False,0,Graduate,False,3865,1640.0,NaN,360.0,1.0,Rural,True,5505.0
2,284,LP001922,Male,True,0,Graduate,False,20667,0.0,NaN,360.0,1.0,Rural,False,20667.0
3,322,LP002054,Male,True,2,Not Graduate,False,3601,1590.0,NaN,360.0,1.0,Rural,True,5191.0
4,231,LP001768,Male,True,0,Graduate,<NA>,3716,0.0,42.0,180.0,1.0,Rural,True,3716.0


In [20]:
def get_embedding_for_col(column):
  embeddings = []
  #print(f'{column=}')
  for row in column:
    embeddings.append(row)
  return embeddings

def get_embeddings_for_table(table):
  embeddings = []
  for col in table.columns:
    embeddings.append(get_embedding_for_col(table[col]))
  return embeddings


def text_embedding(text):
    """Text embedding with a Large Language Model."""
    model = TextEmbeddingModel.from_pretrained("textembedding-gecko@001")
    embeddings = model.get_embeddings(text)
    for embedding in embeddings:
        vector = embedding.values
        print(f"Length of Embedding Vector: {len(vector)}")
    return vector

In [28]:
def get_embedding_for_col2(column):
  print(f'{column=}')
  #convert column to list of strings
  col_strings = [str(cell) for cell in column]
  #convert list of strings to string
  col_string = ' '.join(col_strings)
  print(f'{col_string=}')
  embeddings = text_embedding([col_string])
  return embeddings

def get_embeddings_for_table2(table):
  embeddings = []
  for col in table.columns:
    embeddings.append(get_embedding_for_col2(table[col]))
  return embeddings

In [29]:
embeddings_df1 = pd.DataFrame(get_embeddings_for_table2(table1_df))

embeddings_df1.head()

column=0       18
1       18
2       18
3       18
4       18
        ..
1333    64
1334    64
1335    64
1336    64
1337    64
Name: age, Length: 1338, dtype: Int64
col_string='18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 18 20 20 20 20 20 20 21 21 21 21 21 21 21 22 22 22 22 22 22 22 23 23 23 23 23 23 23 24 24 24 24 24 24 24 25 25 25 25 25 25 25 26 26 26 26 26 26 26 27 27 27 27 27 27 27 28 28 28 28 28 28 28 29 29 29 29 29 29 29 30 30 30 30 30 30 31 31 31 31 31 31 31 32 32 32 32 32 33 33 33 33 33 34 34 34 34 34 34 34 35 35 35 35 35 35 36 36 36 36 36 36 36 37 37 37 37 37 37 38 38 38 38 38 38 39 39 39 39 39 39 40 40 40 40 40 40 40 41 41 41 41 41 41 42 42 42 42 42 42 43 43 43 43 43 43 43 44 44 44 44 44 44 45 45 45 45 45 45 45 46 46 46 46 46 46 46 46 47 47 47 47 47 47 47 48 48 48 48 48 48 48 49 49 49 49 49 49 49 50 50 50 50 50 50 50 51 51 51 51 51 51 51 52 52 52 52 52 52 52 53 53 53 53 53 53 53 54 54 54 54 54 54 54 55 55 55 55 55 55 56 56 56 

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,-0.000559,-0.035793,-0.009178,0.004524,-0.003829,-0.013980,0.051092,0.015469,-0.014350,0.024686,...,-0.002745,0.074047,0.017401,0.038755,-0.000024,0.005925,0.038116,0.001189,-0.040394,-0.036035
1,0.025472,-0.001743,-0.010759,-0.019205,0.021721,0.008235,0.019129,-0.001784,-0.008285,0.034159,...,-0.014890,0.022447,0.022449,0.012948,-0.006195,0.004859,0.005913,-0.023963,-0.054531,-0.033825
2,0.007519,-0.039445,0.015890,0.028511,-0.032738,-0.072867,0.056345,-0.005824,-0.009888,0.015093,...,-0.014119,0.069494,0.006804,0.016065,-0.012949,0.014691,0.056229,-0.000119,-0.010414,0.012084
3,-0.004866,-0.034565,0.013889,0.022807,0.046700,-0.051136,0.061994,0.019789,0.000900,0.016786,...,-0.003433,0.054836,0.010843,-0.000347,0.006316,0.025159,-0.003742,-0.004269,-0.038281,-0.017022
4,0.018648,0.000114,0.021790,-0.010797,0.040144,-0.009694,0.060976,0.012912,-0.044451,0.007194,...,0.005312,0.045549,0.023230,0.046361,0.003600,0.039303,-0.004700,-0.016437,-0.047184,-0.051305
